# Phase 1 — Notebook 2: TF-IDF, Category Analysis & NMF

Builds feature matrices and runs all Phase 1 analytical work on the clean
corpus from `01_preprocess.ipynb`. `make_vec`, `add_bin`, and `group_key` —
all imported from `utils.py` — are shared across Steps 1, 3, and 4, which is
the main reason those three live together.

| Step | What | Key output |
|------|------|------------|
| 1 | Build TF-IDF matrices | `features/X_*.npz`, `vocab/vocab_*.csv` |
| 2 | Quality CP2 | `quality/quality_cp2.json` |
| 3 | Category TF-IDF | `analysis/category_tfidf.csv` |
| 4 | NMF topic discovery | `analysis/nmf_topics.csv`, `nmf_weights.csv` |
| 5 | LLM topic labeling | `analysis/llm_topic_labels.json` |


## Setup

In [1]:
import sys, json, re, unicodedata
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
import time as _time
import openai as _openai

import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.decomposition import NMF
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH

ROOT = Path(".")
sys.path.insert(0, str(ROOT))
from utils import (load_cfg, tokens_to_str, make_vec, add_bin, group_key,
                   quality_report, build_output_path, get_run_date,
                   get_llm_client, build_project_topic_bridge,
                   slugify_group_value)

CFG           = load_cfg()
ct            = CFG["tfidf"]
RUN_DATE      = get_run_date()
GROUPBY_FIELD = CFG["analysis"]["group_by"]
client        = get_llm_client()

def OUT(subdir, fname):
    """Analysis outputs nest under OUTPUTS/{GROUPBY_FIELD}/{RUN_DATE}/"""
    return build_output_path(subdir, fname,
                             groupby_field=GROUPBY_FIELD, run_date=RUN_DATE)

df = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_enriched.parquet")
print(f"Loaded {len(df):,} projects  |  {df.shape[1]} columns")
print(f"Output path: OUTPUTS/{GROUPBY_FIELD}/{RUN_DATE}/")


Loaded 896,277 projects  |  45 columns
Output path: OUTPUTS/project_category/2026-03-27/


---
## Parameters

All tunable values live here. Edit this cell; do not edit parameter references in downstream cells.


In [2]:
# ── All parameters from params.yaml — edit there, not here ──────────────
GROUPBY_FIELD       = CFG["analysis"]["group_by"]
GROUP_DESCRIPTION   = CFG["analysis"]["group_descriptions"].get(
                          GROUPBY_FIELD,
                          f"groups defined by '{GROUPBY_FIELD}' in DonorsChoose data")
MIN_SHARED          = CFG["analysis"]["nmf_min_shared"]
MIN_COVERAGE        = CFG["analysis"]["min_coverage"]
N_COMPONENTS        = CFG["nmf"]["n_components"]
CAT_TFIDF_TOP_N     = CFG["analysis"]["cat_tfidf_top_n"]
N_REPRESENTATIVE    = CFG["llm"]["n_representative_snippets"]
TOP_TERMS_IN_PROMPT = CFG["llm"]["top_terms_in_prompt"]
REVIEW_GROUP        = CFG["analysis"]["review_group"]
EXCLUDE_GROUPS      = CFG["analysis"]["exclude_groups"]
MAX_WORKERS         = CFG["analysis"]["synthesis_max_workers"]
MODEL_LABELING      = CFG["models"]["labeling"]
MODEL_SYNTHESIS     = CFG["models"]["synthesis"]

bins       = CFG["analysis"].get("bins", [])
group_cols = [GROUPBY_FIELD, "bin"] if bins else [GROUPBY_FIELD]

print(f"GROUPBY_FIELD    = {GROUPBY_FIELD!r}")
print(f"EXCLUDE_GROUPS   = {EXCLUDE_GROUPS}")
print(f"MAX_WORKERS      = {MAX_WORKERS}")


GROUPBY_FIELD    = 'project_category'
EXCLUDE_GROUPS   = []
MAX_WORKERS      = 8


---
## Step 1 — Build TF-IDF Matrices

One vectorizer fit per n-gram range on the full corpus. Trigrams are persisted
even if unused in Phase 1 analysis — Phase 2+ inherits them without re-extraction.


In [3]:
# Cache once — reused by Cells 10 and 13
docs  = df["tokens"].apply(tokens_to_str).tolist()
# N-grams are generated here by sklearn rather than in preprocessing.
# The vocab CSVs written below are the persisted n-gram vocabulary the
# spec calls for — just produced at feature time instead of as a
# separate preprocessing step, which avoids a redundant intermediate artifact.
specs = {"X_unigram": (1,1), "X_bigram": (2,2), "X_unigram_bigram": (1,2)}
if CFG["ngrams"]["max_n"] >= 3:
    specs["X_trigram"] = (3, 3)

matrices, vecs = {}, {}
for name, rng in specs.items():
    vec            = make_vec(ct["min_df"], ct["max_df"], rng)
    matrices[name] = vec.fit_transform(docs)
    vecs[name]     = vec
    sz, nnz        = matrices[name].shape, matrices[name].nnz
    print(f"  {name:20s}: shape={sz}  sparsity={1 - nnz/(sz[0]*sz[1]):.3f}")

# Vocab artifact: feature names + IDF weights per matrix
for name, vec in vecs.items():
    (pd.DataFrame({"feature": vec.get_feature_names_out(), "idf": vec.idf_})
       .sort_values("idf")
       .to_csv(OUT("vocab", f"vocab_{name}.csv"), index=False))

feat_dir = ROOT / "OUTPUTS/features"
feat_dir.mkdir(parents=True, exist_ok=True)
for name, X in matrices.items():
    sp.save_npz(feat_dir / f"{name}.npz", X.tocsr())

META_COLS = ["project_id", "project_category", "posted_date", "funded_date",
             "metro_type_at_time_of_posting", "grade_band", "theme_version"]
meta = df[META_COLS].reset_index(drop=True)
try:
    meta.to_parquet(feat_dir / "meta.parquet", index=False)
except ImportError:
    meta.to_csv(feat_dir / "meta.csv", index=False)

assert matrices["X_unigram"].shape[0] == len(df)


  X_unigram           : shape=(896277, 7296)  sparsity=0.992
  X_bigram            : shape=(896277, 932870)  sparsity=1.000
  X_unigram_bigram    : shape=(896277, 940166)  sparsity=1.000
  X_trigram           : shape=(896277, 691995)  sparsity=1.000


---
## Step 2 — Quality Report: Checkpoint 2


In [4]:
qr2 = quality_report(df, label="cp2", matrices=matrices,
                     save_path=OUT("quality", "quality_cp2.json"))


=======================================================  [cp2]
  Projects : 896,277
  Tok/proj : min=5  p50=62  max=89
  Vocab    : 7,389 unique tokens
  X_unigram           : shape=[896277, 7296]  sparsity=0.992
  X_bigram            : shape=[896277, 932870]  sparsity=1.000
  X_unigram_bigram    : shape=[896277, 940166]  sparsity=1.000
  X_trigram           : shape=[896277, 691995]  sparsity=1.000
  Stopwords: FAIL — ['project', 'grade', 'teacher', 'education']



---
## Step 3 — Category TF-IDF

The vectorizer is fit **once** on the full corpus; category slices are scored
by index. This avoids the string-comparison bug (identical token sets across
projects would misclassify rows) and is much faster than refitting per slice.

**Contrast** = token prevalence in this category minus prevalence outside it.

Time bins: defined in `params.yaml` under `analysis.bins`; leave empty for
the full date range.


In [5]:
top_n     = CAT_TFIDF_TOP_N
min_proj  = CFG["analysis"]["min_group_projects"]

df_work   = add_bin(df, bins) if bins else df.copy()
df_work   = df_work.reset_index(drop=True)
all_docs  = (df_work["tokens"].apply(tokens_to_str).tolist() if bins else docs)
vec_cat   = make_vec(ct["min_df"], ct["max_df"], tuple(ct["ngram_range"]))
X_full    = vec_cat.fit_transform(all_docs)
feat      = vec_cat.get_feature_names_out()
idf_vals  = vec_cat.idf_

def cat_tfidf_slice(idx):
    """Score category slice by index against the rest of the corpus."""
    rest_idx  = df_work.index.difference(idx)
    X_cat     = X_full[idx.tolist()]
    X_rest    = X_full[rest_idx.tolist()]
    cat_prev  = (X_cat  > 0).mean(axis=0).A1
    rest_prev = (X_rest > 0).mean(axis=0).A1 if len(rest_idx) else np.zeros(len(feat))
    tf        = X_cat.mean(axis=0).A1
    return (pd.DataFrame({
                "token":         feat,
                "tf":            tf,
                "idf":           idf_vals,
                "tfidf":         tf * idf_vals,
                "prevalence":    cat_prev,
                "contrast":      cat_prev - rest_prev,
                "project_count": (X_cat > 0).sum(axis=0).A1.astype(int),
            }).nlargest(top_n, "tfidf"))

rows = []
for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj:
        continue
    kd  = group_key(keys, group_cols)
    top = cat_tfidf_slice(sub.index)
    for col, val in kd.items():
        top.insert(0, col, val)
    rows.append(top)

if not rows:
    raise RuntimeError(f"No groups met min_proj={min_proj} threshold — lower min_group_projects in params.yaml")
cat_tfidf_df = pd.concat(rows, ignore_index=True)
cat_tfidf_df.to_csv(OUT("analysis", "category_tfidf.csv"), index=False)
print(f"{len(cat_tfidf_df):,} rows  |  {cat_tfidf_df[group_cols[0]].nunique()} groups")
cat_tfidf_df.head(10)


300 rows  |  15 groups


,project_category,token,tf,idf,tfidf,prevalence,contrast,project_count
0,Art Supplies,art,0.021509,3.258761,0.070091,0.465726,0.381497,22156
1,Art Supplies,paint,0.014017,4.777631,0.066968,0.207513,0.194987,9872
2,Art Supplies,supply,0.016237,2.782164,0.045174,0.406386,0.251460,19333
3,Art Supplies,color,0.011566,3.724483,0.043077,0.218632,0.161633,10401
4,Art Supplies,creativity,0.010943,3.693527,0.040420,0.216783,0.157502,10313
5,Art Supplies,marker,0.009618,4.082559,0.039267,0.163811,0.124583,7793
6,Art Supplies,express,0.009308,4.126950,0.038412,0.162887,0.125709,7749
7,Art Supplies,cricut,0.005896,6.350626,0.037446,0.068568,0.067402,3262
8,Art Supplies,creative,0.011658,3.156758,0.036803,0.268303,0.161159,12764
9,Art Supplies,crayon,0.007216,5.072622,0.036602,0.098480,0.086014,4685


---
## Step 4 — NMF Topic Discovery

NMF is fit independently per category so the dominant axis in one category
does not suppress signal in others. Topics are candidates for human review —
not stable theme definitions (that is Phase 2).

The weights DataFrame records which projects load most strongly on each topic;
Step 5 uses it to select representative snippets by NMF weight rather than
randomly.


In [6]:
cn    = CFG["nmf"]
n_rep = N_REPRESENTATIVE
min_proj = CFG["analysis"]["min_group_projects"]

def nmf_one(docs):
    """Fit NMF on one slice. Returns (topics_df, W) or (None, None)."""
    vec = make_vec(ct["min_df"], ct["max_df"], tuple(ct.get("ngram_range", [1, 1])))
    X   = vec.fit_transform(docs)
    if X.shape[1] < N_COMPONENTS:
        return None, None
    model = NMF(n_components=N_COMPONENTS, random_state=cn["random_seed"],
                init="nndsvd", max_iter=cn["max_iter"])
    W     = model.fit_transform(X)
    vocab = vec.get_feature_names_out()
    rows  = []
    for i, comp in enumerate(model.components_):
        idx = comp.argsort()[::-1][:cn["top_words"]]
        rows.append({"topic_id": i, "top_terms": vocab[idx].tolist(),
                     "top_weights": comp[idx].tolist()})
    return pd.DataFrame(rows), W

all_topics, all_weights = [], []
df_work = add_bin(df, bins) if bins else df.copy()
df_work = df_work.reset_index(drop=True)

for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj:
        continue
    kd     = group_key(keys, group_cols)
    group_docs = sub["tokens"].apply(tokens_to_str).tolist()
    pids   = sub["project_id"].tolist()
    topics, W = nmf_one(group_docs)
    if topics is None:
        continue
    for col, val in kd.items():
        topics[col] = val
    all_topics.append(topics)
    for tid in range(W.shape[1]):
        order = W[:, tid].argsort()[::-1]
        for rank, idx in enumerate(order):
            all_weights.append({
                **kd,
                "topic_id": tid,
                "project_id": pids[idx],
                "weight": float(W[idx, tid]),
                "rank": rank,
            })

if not all_topics:
    raise RuntimeError(
        f"No groups produced NMF topics — check N_COMPONENTS={N_COMPONENTS} vs vocab size, "
        f"or lower min_group_projects"
    )

topics_df  = pd.concat(all_topics, ignore_index=True)
weights_df = pd.DataFrame(all_weights)
topics_df.to_csv(OUT("analysis", "nmf_topics.csv"),   index=False)
weights_df.to_csv(OUT("analysis", "nmf_weights.csv"), index=False)
print(f"{len(topics_df):,} topics across {topics_df[group_cols[0]].nunique()} groups")

# ── Build project-topic bridge ───────────────────────────────────────────
# weights_df must contain the FULL per-project topic-weight matrix for each
# retained group before any top-k or representative-project filtering. If the
# notebook currently reduces weights_df earlier, build the bridge from the raw
# NMF W matrix before that reduction step.
threshold = CFG["analysis"]["topic_assignment_threshold"]
project_topic_bridge_df = build_project_topic_bridge(
    weights_df, GROUPBY_FIELD, threshold
)
project_topic_bridge_df.to_csv(
    OUT("analysis", "project_topic_bridge.csv"), index=False
)
print(
    f"Bridge: {len(project_topic_bridge_df):,} project-topic assignments "
    f"(topic_share >= {threshold})"
)

topics_df.head(6)


180 topics across 15 groups
Bridge: 741,852 project-topic assignments (topic_share >= 0.3)


,topic_id,top_terms,top_weights,project_category
0,0,"[hard, some, thing, one, work, love, kid, clas...","[0.7794189762826691, 0.7597492401240165, 0.741...",Art Supplies
1,1,"[problem, solve, problem solve, thinke, hand, ...","[0.761300615978416, 0.7311979164194524, 0.6363...",Art Supplies
2,2,"[paint, art, artist, medium, create, artwork, ...","[0.9450864449513288, 0.9218133804597908, 0.644...",Art Supplies
3,3,"[motor, fine, fine motor, motor skill, skill, ...","[1.3406051544110913, 1.2862956382237984, 1.257...",Art Supplies
4,4,"[language, english, learner, diverse, english ...","[1.2126182261101017, 1.0916055085906375, 0.867...",Art Supplies
5,5,"[pencil, marker, crayon, supply, glue, color, ...","[1.182659150886908, 1.0838385523517313, 0.9053...",Art Supplies


In [7]:
# Cross-group universal themes — terms appearing in NMF topics across many groups
if REVIEW_GROUP is not None:
    _eligible = topics_df[group_cols[0]].unique().tolist()
    if REVIEW_GROUP not in _eligible:
        raise ValueError(
            f"REVIEW_GROUP={REVIEW_GROUP!r} not found. Eligible: {_eligible}"
        )
else:
    REVIEW_GROUP = df_work.groupby(GROUPBY_FIELD).size().idxmax()

records = [(r[group_cols[0]], frozenset(r.top_terms)) for _, r in topics_df.iterrows()]

theme_cats = defaultdict(set)
for i, (ci, si) in enumerate(records):
    for cj, sj in records[i+1:]:
        if ci == cj:
            continue
        shared = si & sj
        if len(shared) >= MIN_SHARED:
            key = frozenset(shared)
            theme_cats[key] |= {ci, cj}

# Sort by n_groups desc, then drop any theme whose terms are a subset of a prior theme
rows = sorted(theme_cats.items(), key=lambda x: -len(x[1]))
seen = []
deduped = []
for terms, cats in rows:
    if len(cats) < MIN_COVERAGE:
        continue
    if not any(terms <= prior for prior in seen):
        deduped.append({"theme": ", ".join(sorted(terms)[:5]),
                        "n_groups": len(cats),
                        "categories": sorted(cats)})
        seen.append(terms)

pd.DataFrame(deduped).reset_index(drop=True)


,theme,n_groups,categories
0,"income, low, low income",14,"[Art Supplies, Books, Classroom Basics, Comput..."
1,"family, income, low, low income",11,"[Art Supplies, Books, Classroom Basics, Comput..."
2,"family, home, income, low, low income",10,"[Art Supplies, Books, Classroom Basics, Comput..."
3,"free, free reduce, lunch, receive, receive free",10,"[Art Supplies, Classroom Basics, Computers & T..."
4,"free, free reduce, lunch, price, receive",10,"[Art Supplies, Classroom Basics, Computers & T..."
5,"breakfast, free, free reduce, lunch, price",9,"[Art Supplies, Books, Classroom Basics, Comput..."
6,"breakfast, free, free reduce, lunch, price",8,"[Art Supplies, Classroom Basics, Computers & T..."
7,"challenge, face, family, income, low",7,"[Art Supplies, Classroom Basics, Educational K..."
8,"english, english language, language learner",6,"[Art Supplies, Books, Computers & Tablets, Ins..."
9,"emotional, self, social",6,"[Art Supplies, Books, Computers & Tablets, Fle..."


---
## Step 5 — LLM Topic Labeling

One API call per topic on compressed input only — never raw essay text at scale.
Representative snippets are chosen by NMF weight (not random).

Prompt field limits (`top_unigrams`, `top_bigrams`, `top_nmf_terms`) are applied
as item counts. Parse failures are stored with enough metadata to debug later.

**Requires:** `pip install anthropic` + `ANTHROPIC_API_KEY` env var.
Without the package, stubs are written so downstream steps stay testable.


In [8]:
SYSTEM = (
    "You are an NLP analyst reviewing NMF topic clusters from DonorsChoose teacher "
    "essays. Respond ONLY with a JSON object, no preamble, no markdown fences.\n\n"
    "Token naming conventions you may see in topic terms:\n"
    "  __framing_[name]__ = a rhetorical framing signal injected during preprocessing "
    "(e.g. __framing_barrier_removal__ means essays removing obstacles to participation)\n"
    "  __cat_[name]__ = a subject matter category token (e.g. __cat_genetics__)\n"
    "  __sub_[name]__ = a subcategory token\n"
    "These are meaningful signals — incorporate them into your label and description.\n\n"
    "Labeling rules:\n"
    "- Prefer the most specific defensible label.\n"
    "- Preserve concrete signals such as named programs, vendors, pedagogies, student "
    "populations, classroom use cases, or rhetorical framing when present.\n"
    "- Do not collapse topics into broad umbrella labels like technology, literacy, "
    "engagement, classroom supplies, or social-emotional learning if the evidence "
    "supports a narrower interpretation.\n"
    "- When framing tokens are present, treat them as first-class evidence about how "
    "teachers are positioning the request, not just background metadata.\n"
    "- If a topic is mixed, say what subthemes are colliding rather than forcing an "
    "overly neat label."
)
PROMPT = (
    f"Group ({GROUPBY_FIELD}): {{group_value}}{{bin_line}}\n"
    "Topic {topic_id}\n"
    "Top unigrams : {unigrams}\n"
    "Top bigrams  : {bigrams}\n"
    "Top NMF terms: {nmf_terms}\n"
    "Representative project tokens:\n"
    "{snippets}\n\n"
    "Instructions:\n"
    "- proposed_label should be concrete and specific, ideally naming the request or "
    "intervention, the population or context, and the framing when supported.\n"
    "- description should explain what makes this topic distinct from nearby generic "
    "topics.\n"
    "- If the topic appears broad, identify the narrower mechanism, use case, or "
    "population if the evidence supports it.\n"
    "- If the evidence is genuinely mixed, preserve that ambiguity instead of inventing "
    "a falsely precise label.\n\n"
    f"Return a JSON object with exactly these keys:\n"
    f"  {GROUPBY_FIELD}, topic_id, proposed_label, description, coherence_flag, notes\n"
    'coherence_flag must be one of: "coherent", "mixed", "redundant"'
)

has_essay = "essay" in df.columns
pid_text  = (df.set_index("project_id")["essay"].str[:300] if has_essay
             else df.set_index("project_id")["tokens"].apply(lambda t: " ".join(t[:40])))

def build_input(t_row):
    terms = t_row["top_terms"]
    key_cols = [GROUPBY_FIELD] + (["bin"] if "bin" in t_row.index else [])
    mask = weights_df["topic_id"] == t_row["topic_id"]
    for col in key_cols:
        mask &= weights_df[col] == t_row[col]
    rep_pids = (weights_df[mask].sort_values("weight", ascending=False)
                    ["project_id"].tolist()[:N_REPRESENTATIVE])
    n_uni = TOP_TERMS_IN_PROMPT
    n_bi  = max(2, TOP_TERMS_IN_PROMPT // 2)
    n_nmf = TOP_TERMS_IN_PROMPT
    return {
        "group_value": t_row[GROUPBY_FIELD],
        "topic_id": int(t_row["topic_id"]),
        "bin_line": f"\nBin: {t_row['bin']}"
                    if "bin" in t_row.index and pd.notna(t_row.get("bin")) else "",
        "unigrams":  ", ".join([x for x in terms if " " not in x][:n_uni]),
        "bigrams":   ", ".join([x for x in terms if " " in  x][:n_bi]),
        "nmf_terms": ", ".join(terms[:n_nmf]),
        "snippets":  "\n".join(f"- {pid_text.get(p, '')}" for p in rep_pids),
    }

results = []
for _, t in topics_df.iterrows():
    inp  = build_input(t)
    resp = client.chat.completions.create(
        model=MODEL_LABELING,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": PROMPT.format(**inp)},
        ],
    )
    text = resp.choices[0].message.content.strip()
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        obj = {"raw": text, "parse_error": True, "model": MODEL_LABELING,
               "timestamp": datetime.now().isoformat(),
               GROUPBY_FIELD: inp["group_value"], "topic_id": inp["topic_id"]}
    results.append(obj)
    print(f"  {inp['group_value']} / topic {inp['topic_id']} "
          f"→ {obj.get('proposed_label', '?')}")

parse_errors = [r for r in results if r.get("parse_error")]
if parse_errors:
    print(f"WARNING: {len(parse_errors)} topics failed JSON parse — excluded from synthesis:")
    for e in parse_errors:
        print(f"  {e.get(GROUPBY_FIELD,'?')} / topic {e.get('topic_id','?')}")

labels_df = pd.DataFrame([r for r in results if not r.get("parse_error")])

assert GROUPBY_FIELD in labels_df.columns, (
    f"labels_df missing '{GROUPBY_FIELD}' — re-run labeling for this groupby field"
)

with open(OUT("analysis", "llm_topic_labels.json"), "w") as f:
    json.dump(results, f, indent=2)
print(f"\n{len(results)} labels saved")


  Art Supplies / topic 0 → Creative Engagement Through Art for Second Graders
  Art Supplies / topic 1 → STEM Makerspace Projects for Critical Thinking and Creativity
  Art Supplies / topic 2 → Art Medium Exploration and Technique Development for Visual Arts Education
  Art Supplies / topic 3 → Fine Motor Skill Development for Early Childhood through Sensory Play
  Art Supplies / topic 4 → Art Supplies for English Language Learners from Diverse Backgrounds
  Art Supplies / topic 5 → Basic Art Supply Replenishment for Students' Classroom Needs
  Art Supplies / topic 6 → Social Emotional Learning Resources for Stress Regulation
  Art Supplies / topic 7 → Cricut Machine Use for Personalized Art Projects in Classroom Settings
  Art Supplies / topic 8 → Literacy and Math Engagement Activities for Diverse Kindergarten Classrooms
  Art Supplies / topic 9 → Creative Resources for Low-Income Students Facing Socioeconomic Challenges
  Art Supplies / topic 10 → Art Supply Requests for Students Re

In [9]:
labels_df = pd.DataFrame(results)
labels_df.groupby("coherence_flag").size().rename("count").to_frame()
labels_df[[GROUPBY_FIELD,"topic_id","proposed_label","coherence_flag","description"]]

,project_category,topic_id,proposed_label,coherence_flag,description
0,Art Supplies,0,Creative Engagement Through Art for Second Gra...,coherent,This topic centers on the enthusiasm and effor...
1,Art Supplies,1,STEM Makerspace Projects for Critical Thinking...,coherent,This topic centers around projects aimed at fo...
2,Art Supplies,2,Art Medium Exploration and Technique Developme...,coherent,This topic centers around requests for various...
3,Art Supplies,3,Fine Motor Skill Development for Early Childho...,coherent,This topic focuses on interventions aimed at e...
4,Art Supplies,4,Art Supplies for English Language Learners fro...,coherent,This topic centers around requests for art sup...
...,...,...,...,...,...
175,Virtual Visitors,7,Virtual Author Visits for Reluctant and Avid R...,coherent,This topic focuses on projects that facilitate...
176,Virtual Visitors,8,Virtual Community Building for Holistic Develo...,coherent,This topic focuses on creating virtual experie...
177,Virtual Visitors,9,Support for Deaf and Hard of Hearing English L...,coherent,This topic focuses on initiatives aimed at sup...
178,Virtual Visitors,10,Field Trip Experiences for Rural Second and Fo...,coherent,This topic focuses on enhancing science educat...


In [23]:
# ── SYNTHESIS — cross-group + per-group loops ─────────────────────────────

def clean_label(text):
    """Remove injected __token__ signals and replace with readable paraphrase."""
    return re.sub(r'__[a-z_]+__\s*', '', str(text)).strip()

def build_topic_lines(df, groupby_field, group=None):
    if group is not None:
        df = df[df[groupby_field] == group]
    return "\n".join(
        f"  {row[groupby_field]} | topic {row.topic_id} | "
        f"label: {clean_label(row.proposed_label)} | "
        f"coherence: {row.coherence_flag} | "
        f"description: {clean_label(row.description)}"
        for _, row in df.iterrows()
    )

SYNTHESIS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "You write precise, insight-driven briefings for internal strategy audiences. "
    "You do not pad your answers or state the obvious.\n\n"
    "Some topic labels may reference rhetorical framing signals (e.g. 'barrier removal "
    "framing', 'chronic scarcity language') — these come from injected tokens that "
    "categorize how teachers write, not just what they request. Treat these as "
    "meaningful signals about teacher rhetorical strategy.\n\n"
    "Prioritize nuance over novelty. Prefer fine-grained distinctions, concrete "
    "mechanisms, specific populations, and framing differences over broad thematic "
    "summaries.\n\n"
    "Formatting: use plain numbered section headers (e.g. '1. SUBTHEME DISTINCTIONS'). "
    "Do not use markdown headers (##). Do not use bullet points.\n\n"
    "Grounding: when writing Framing Differences findings, the contrast must be "
    "traceable to explicit language in the topic label or description — not inferred "
    "from the topic's general subject matter or implied by category context. If you "
    "cannot quote or closely paraphrase specific label language to establish both sides "
    "of a contrast, do not make the finding."
)

topic_lines = build_topic_lines(labels_df, GROUPBY_FIELD)

SYNTHESIS_PROMPT_CROSS_GROUP = f"""
Below is a list of NMF topics discovered from teacher project request essays on DonorsChoose,
grouped by {GROUPBY_FIELD} ({GROUP_DESCRIPTION}).
Each topic represents a cluster of essays with similar language, framing, and request patterns.
{topic_lines}

Your task: identify the most specific, decision-useful patterns in this topic landscape.

Rules:
- Every finding must be grounded in specific topic labels, descriptions, or coherence flags
  from the list above.
- Do not introduce concepts, themes, or absences that cannot be traced to an actual topic.
- If you cannot name the specific group and topic, do not make the finding.
- Prefer fine-grained distinctions over broad summaries.
- Skip anything predictable given the group name.
- Skip the free/reduced lunch poverty framing — it appears everywhere and is already known.
- Skip generic engagement/motivation language unless the topic evidence makes it unusually specific.
- If fewer than two strong findings exist for a section, write one or write none. Do not pad.

Focus on:
1. RECURRING INTERNAL SPLITS — cases where the same subtheme distinction recurs independently
   inside multiple groups. Only flag a split if it appears in at least two groups with
   meaningfully similar internal logic. Name the groups and specific topics each time.

2. CROSS-GROUP SIGNATURES — a specific program, vendor, pedagogy, student need, or rhetorical
   frame that shows up as a recognizable topic in multiple groups. Name every group it appears
   in and explain why the pattern is meaningful.

3. FRAMING DIFFERENCES — cases where similar underlying needs are presented through meaningfully
   different rhetorical frames, populations, or classroom contexts. Explain what changes across
   groups and why that difference matters.

4. NOTABLE ABSENCES — a topic you would expect given the internal logic of a group's other
   topics, but that is missing. To qualify, name at least two specific topics already present
   in the same group that make the gap conspicuous from within. Do not reason from other groups.

5. REQUEST TRIGGERS AND PITCH STRATEGIES — patterns in why teachers say they are requesting and
   how they frame those reasons for a donor audience. Only use explicit topic evidence.

Format as five labeled sections. Under each, write 2-5 findings as short paragraphs.
Be specific — name the group and topic label every time. Do not use bullet points.
""".strip()

PER_GROUP_INSTRUCTIONS = """
Your task: identify the most specific, decision-useful patterns within this group.

Rules:
- Every finding must be grounded in specific topic labels, descriptions, or coherence flags
  from the list above.
- Do not introduce concepts, themes, or absences that cannot be traced to an actual topic.
- If you cannot name the specific topic, do not make the finding.
- Prefer fine-grained distinctions over broad summaries.
- Skip anything predictable given the group name.
- Skip the free/reduced lunch poverty framing — it appears everywhere and is already known.
- Skip generic engagement/motivation language unless the topic evidence makes it unusually specific.
- If fewer than two strong findings exist for a section, write one or write none. Do not pad.

Focus on:
1. SUBTHEME DISTINCTIONS — distinct needs, populations, pedagogies, use cases, or framing
   modes within this group.

2. INTERNAL FRAMING DIFFERENCES — cases within this group where similar underlying needs are
   presented through meaningfully different rhetorical frames, populations, or classroom
   contexts.

3. NOTABLE ABSENCES — a topic you would expect given the internal logic of this group's other
   topics, but that is missing.

4. REQUEST TRIGGERS AND PITCH STRATEGIES — patterns in why teachers say they are requesting
   within this group and how they frame that reason for a donor audience.

Format as four labeled sections. Under each, write 2-5 findings as short paragraphs.
Be specific — name the topic label every time. Do not use bullet points.
""".strip()

def build_per_group_prompt(group, topic_lines_text):
    return f"""
Below is a list of NMF topics discovered from teacher project request essays on DonorsChoose
for a single group: {group} ({GROUP_DESCRIPTION}).
Each topic represents a cluster of essays with similar language, framing, and request patterns.
{topic_lines_text}

{PER_GROUP_INSTRUCTIONS}
""".strip()

def _call_with_retry(prompt, max_retries=3):
    """Single synthesis call with exponential backoff on rate-limit/timeout."""
    for attempt in range(max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_SYNTHESIS,
                messages=[
                    {"role": "system", "content": SYNTHESIS_SYSTEM},
                    {"role": "user",   "content": prompt},
                ],
            )
            return resp.choices[0].message.content.strip()
        except (_openai.RateLimitError, _openai.APITimeoutError):
            if attempt < max_retries:
                _time.sleep(2 ** attempt)
            else:
                raise
        except Exception as e:
            print(f"Non-retryable error: {e}")
            return None

synthesis_cross = _call_with_retry(SYNTHESIS_PROMPT_CROSS_GROUP)
if synthesis_cross is None:
    raise RuntimeError("Cross-group synthesis failed; stopping before downstream cells.")

with open(OUT("analysis", "llm_synthesis_cross_group.txt"), "w") as f:
    f.write(synthesis_cross)

print("── CROSS-GROUP ──────────────────────────────────────────")
print(synthesis_cross)

groups = [
    g
    for g in labels_df[GROUPBY_FIELD].dropna().unique().tolist()
    if g not in EXCLUDE_GROUPS
]

def synthesize_one_group(group):
    topic_lines_text = build_topic_lines(labels_df, GROUPBY_FIELD, group=group)
    prompt = build_per_group_prompt(group, topic_lines_text)
    result = _call_with_retry(prompt)
    if result is None:
        return group, None

    slug = slugify_group_value(group)
    fpath = OUT("analysis", f"llm_synthesis_{slug}.txt")
    with open(fpath, "w") as f:
        f.write(result)
    return group, result

per_group_results = {}

total_groups = len(groups)
successful_groups = 0
failed_groups = 0
total_chars = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(synthesize_one_group, g): g for g in groups}

    for future in as_completed(futures):
        submitted_group = futures[future]

        try:
            group, result = future.result()
        except Exception as e:
            failed_groups += 1
            print(f"FAILED: {submitted_group} | error: {e}")
            continue

        if result is not None:
            per_group_results[group] = result
            successful_groups += 1

            char_count = len(result)
            total_chars += char_count

            print(f"Done: {group} | chars: {char_count}")
            if char_count < 500:
                print(f"  WARNING: {group} output is very short")
        else:
            failed_groups += 1
            print(f"FAILED: {group}")

# Preserve original group order
per_group_results = {g: per_group_results[g] for g in groups if g in per_group_results}

print("\n── SYNTHESIS SUMMARY ───────────────────────────────────")
print(f"Total groups: {total_groups}")
print(f"Successful: {successful_groups}")
print(f"Failed: {failed_groups}")
print(f"Total characters generated: {total_chars}")
print(f"Avg chars per successful group: {round(total_chars / max(successful_groups, 1))}")

if not per_group_results:
    raise RuntimeError("No per-group synthesis results were produced; stopping before downstream cells.")


── CROSS-GROUP ──────────────────────────────────────────
1. RECURRING INTERNAL SPLITS

Within flexible seating-adjacent categories, there is a repeated split between autonomy-oriented seating and regulation-oriented seating. In Flexible Seating, topics 0, 1, 9, and 11 all frame seating as student-directed choice or movement (“student autonomy in choice,” “movement-friendly,” “enhanced oxygen flow and core strength,” “student choice”), while topics 5 and 6 shift to “calm down corners” and “comfortable, relaxing learning corners” for special education and emotional respite. The same split appears in Reading Nooks, Desks & Storage, where topic 3 and topic 8 emphasize “collaboration and comfort” and “student choice and autonomy,” while topic 9 is explicitly about “Social-Emotional Learning in Special Education for Students with Autism and Other Disabilities.” This is not just a furniture distinction; it separates requests justified as agency/productivity supports from requests justified a

In [29]:
# ── EXTERNAL-FACING INSIGHTS — structured JSON call ───────────────────────

OUTPUT_GROUP_KEY = "by_group"

# Build required group values dynamically from the current run, preserving order.
REQUIRED_GROUP_VALUES = list(per_group_results.keys())

if not REQUIRED_GROUP_VALUES:
    raise RuntimeError("No synthesized group results are available; cannot build external-facing insights.")

def strip_json_fences(text):
    text = (text or "").strip()
    text = re.sub(r"^\s*```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text)
    return text.strip()

def normalize_source_topics(source_topics):
    """
    Normalize model output to compact 'group|topic_id' strings.
    Accepts either:
      - 'Art Supplies|3'
      - {'group': 'Art Supplies', 'topic_id': 3}
    Returns de-duplicated compact strings.
    """
    out = []

    for src in source_topics or []:
        group = None
        tid = None

        if isinstance(src, str) and "|" in src:
            group, tid = src.rsplit("|", 1)
        elif isinstance(src, dict):
            group = str(src.get("group", "")).strip()
            tid = src.get("topic_id", "")
        else:
            continue

        group = str(group).strip()
        tid = str(tid).strip()

        if not group or not tid:
            continue

        try:
            tid = str(int(tid))
        except Exception:
            continue

        out.append(f"{group}|{tid}")

    return list(dict.fromkeys(out))

def normalize_insight(insight):
    """
    Keep only the fields the downstream cells actually need.
    Drop redundant/invalid keys like 'section' or stray coverage notes.
    """
    if not isinstance(insight, dict):
        raise ValueError(f"Insight must be a dict, got {type(insight)}")

    normalized = {
        "title": str(insight.get("title", "")).strip(),
        "what_seeing": str(insight.get("what_seeing", "")).strip(),
        "why_easy_to_miss": str(insight.get("why_easy_to_miss", "")).strip(),
        "what_it_means": str(insight.get("what_it_means", "")).strip(),
        "source_topics": normalize_source_topics(insight.get("source_topics", [])),
    }

    if not normalized["title"]:
        raise ValueError(f"Insight missing title: {insight}")

    return normalized

synthesis_parts = []
if synthesis_cross:
    synthesis_parts.append(f"=== CROSS-GROUP ANALYSIS ===\n{synthesis_cross}")

for group_value, text in per_group_results.items():
    if text:
        synthesis_parts.append(f"=== {group_value} ===\n{text}")

synthesis_input = "\n\n".join(synthesis_parts)

INSIGHTS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "You write precise, decision-facing briefings for external audiences: "
    "foundations, corporate partners, policymakers, and major donors. "
    "You return ONLY valid JSON. No preamble, no explanation, no markdown fences."
)

INSIGHTS_PROMPT = f"""
You are given a comprehensive structured analysis of classroom project request patterns including:
within-group subthemes, cross-group signatures, framing differences, notable absences,
and request triggers and pitch strategies.

The analysis is grouped by the field "{GROUPBY_FIELD}" ({GROUP_DESCRIPTION}).

{synthesis_input}

Your task: produce a decision-facing insights document for external stakeholders —
executives, foundation leaders, state legislators, and major donors — who are
intelligent but not close to classroom realities. These readers should finish each
insight thinking: "I never would have known that, and the fact that this organization
can see it means I should take their priorities seriously."

SELECTION CRITERIA:
- Each insight must surface something genuinely non-obvious about what classrooms need,
  reframe how a thoughtful outsider should interpret teacher demand, and have a clear
  implication for where attention or resources should go.
- Prioritize insights that reveal hidden scale, misallocated attention, or a shift in
  how teachers are operating that external stakeholders would not expect.
- Avoid methodological commentary, insights that only matter to data practitioners,
  and observations that merely restate what the group label already suggests.

COUNT + COVERAGE REQUIREMENTS:
- Return exactly 10 cross-group insights in "key_insights".
- Return exactly 4 secondary insights in "appendix".
- "{OUTPUT_GROUP_KEY}" must include every group value in this exact list:
  {json.dumps(REQUIRED_GROUP_VALUES, ensure_ascii=False)}
- Return exactly 1 strongest defensible insight for each group value above.
- Do not omit group values silently.
- Prefer slightly weaker but still defensible coverage over omission.

INSIGHT STRUCTURE — for every insight use exactly these fields:
- title: a crisp, declarative headline that makes the finding feel like a reveal,
  not a description.
- what_seeing: 2-4 sentences describing the observed pattern in plain, concrete language.
  Write about what teachers are actually doing or asking for — not about the analysis.
- why_easy_to_miss: 1-2 sentences explaining why this pattern is invisible to someone
  looking at surface-level groups or conventional reporting.
- what_it_means: 2-4 sentences translating the pattern into consequence.
  This should land like a strategic implication, not an analytical note.
- source_topics: a list of the strongest grounding topics for the insight.
- Use only topics that directly support the claim.
- Usually 2-4 topics is enough, but do not force a fixed count.
- Prefer fewer well-matched topics over padding with weak ones.
Example: ["Art Supplies|3", "Books|7"]
  Use the exact group value as it appears in the analysis above.
- Do not include a "section" field inside each insight.
  Section is implied by where the insight appears in the JSON.

STYLE:
- Direct, confident, and human — not academic, not corporate.
- No mention of "topics", "model", "NMF", "pipeline", "cluster", or analytical machinery.
- No hedging phrases like "may suggest" or "could indicate".
- No filler.
- Do not repeat the same implication across insights.

VALIDITY RULES:
- Return valid JSON only.
- The response is incomplete if any required group value is missing from "{OUTPUT_GROUP_KEY}".
- Do not return markdown fences or explanatory text.

Return a JSON object with exactly this structure:
{{
  "key_insights": [/* exactly 10 insight objects */],
  "appendix": [/* exactly 4 insight objects */],
  "{OUTPUT_GROUP_KEY}": {{
    "<group value>": [/* exactly 1 insight object */],
    ...
  }}
}}
""".strip()

resp = client.chat.completions.create(
    model=MODEL_SYNTHESIS,
    messages=[
        {"role": "system", "content": INSIGHTS_SYSTEM},
        {"role": "user",   "content": INSIGHTS_PROMPT},
    ],
    response_format={"type": "json_object"},
)

raw_json = strip_json_fences(resp.choices[0].message.content)
insights_data = json.loads(raw_json)

if not isinstance(insights_data, dict):
    raise ValueError("Top-level insights response is not a JSON object.")

raw_key = insights_data.get("key_insights", [])
raw_appendix = insights_data.get("appendix", [])
raw_by_group = insights_data.get(OUTPUT_GROUP_KEY, {})

if not isinstance(raw_key, list):
    raise ValueError("key_insights is not a list.")
if not isinstance(raw_appendix, list):
    raise ValueError("appendix is not a list.")
if not isinstance(raw_by_group, dict):
    raise ValueError(f"{OUTPUT_GROUP_KEY} is not an object.")

normalized = {
    "key_insights": [normalize_insight(i) for i in raw_key],
    "appendix": [normalize_insight(i) for i in raw_appendix],
    OUTPUT_GROUP_KEY: {},
}

missing_group_values = [g for g in REQUIRED_GROUP_VALUES if g not in raw_by_group]
if missing_group_values:
    raise ValueError(f"Model omitted required group values: {missing_group_values}")

extra_group_values = [g for g in raw_by_group.keys() if g not in REQUIRED_GROUP_VALUES]
if extra_group_values:
    print(f"WARNING: extra group values returned and ignored: {extra_group_values}")

for group_value in REQUIRED_GROUP_VALUES:
    items = raw_by_group.get(group_value, [])
    if not isinstance(items, list):
        raise ValueError(f"{OUTPUT_GROUP_KEY}['{group_value}'] is not a list.")
    normalized[OUTPUT_GROUP_KEY][group_value] = [normalize_insight(i) for i in items]

insights_data = normalized

# Fail fast on under-production instead of silently continuing with a thin object.
if len(insights_data["key_insights"]) != 10:
    raise ValueError(f"Expected 10 key insights, got {len(insights_data['key_insights'])}")
if len(insights_data["appendix"]) != 4:
    raise ValueError(f"Expected 4 appendix insights, got {len(insights_data['appendix'])}")

bad_group_counts = {
    group_value: len(items)
    for group_value, items in insights_data[OUTPUT_GROUP_KEY].items()
    if len(items) != 1
}
if bad_group_counts:
    raise ValueError(f"Expected exactly 1 insight per group value, got: {bad_group_counts}")

with open(OUT("analysis", "insights_structured.json"), "w") as f:
    json.dump(insights_data, f, indent=2)

total_source_topics = (
    sum(len(i.get("source_topics", [])) for i in insights_data["key_insights"]) +
    sum(len(i.get("source_topics", [])) for i in insights_data["appendix"]) +
    sum(len(i.get("source_topics", [])) for items in insights_data[OUTPUT_GROUP_KEY].values() for i in items)
)

print(f"Key insights:  {len(insights_data.get('key_insights', []))}")
print(f"Appendix:      {len(insights_data.get('appendix', []))}")
print(f"Group values:  {list(insights_data.get(OUTPUT_GROUP_KEY, {}).keys())}")
print(f"Total source_topics across all insights: {total_source_topics}")


Key insights:  10
Appendix:      4
Group values:  ['Art Supplies', 'Awaiting Classification', 'Books', 'Classroom Basics', 'Computers & Tablets', 'Educational Kits & Games', 'Flexible Seating', 'Food, Clothing & Hygiene', 'Instructional Technology', 'Lab Equipment', 'Musical Instruments', 'Reading Nooks, Desks & Storage', 'Sports & Exercise Equipment', 'Virtual Trips', 'Virtual Visitors']
Total source_topics across all insights: 131


In [30]:
VERIFY_SYSTEM = (
    "You are checking whether topic labels correctly support a given insight. "
    "Be strict. If the topic is only loosely related, drop it. "
    "Return only valid JSON."
)

def verify_source_topics(insight, labels_df, groupby_field):
    title         = insight.get("title", "")
    what_seeing   = insight.get("what_seeing", "")
    what_it_means = insight.get("what_it_means", "")
    source_topics = insight.get("source_topics", [])

    if not source_topics:
        return insight

    topic_lines = []
    for src in source_topics:
        if isinstance(src, dict):
            group = src.get("group", "")
            tid   = str(src.get("topic_id", ""))
        elif isinstance(src, str) and "|" in src:
            group, tid = src.rsplit("|", 1)
        else:
            continue

        match = labels_df[
            (labels_df[groupby_field] == group) &
            (labels_df["topic_id"]    == int(tid))
        ]
        if not match.empty:
            row = match.iloc[0]
            topic_lines.append(
                f"  {group}|{tid} | label: {row['proposed_label']} | "
                f"description: {row['description']}"
            )

    if not topic_lines:
        return insight

    prompt = f"""
Insight title: {title}

What we're seeing: {what_seeing}

What it means: {what_it_means}

Claimed source topics:
{chr(10).join(topic_lines)}

For each claimed source topic, decide: does this topic label and description
directly support the insight above, or is it only loosely related or irrelevant?

Rules:
- Keep a topic only if its label or description contains language that
  directly grounds a specific claim in the insight.
- Drop it if the connection requires inference, analogy, or category-level association.
- Do not add new topics.

Return JSON: {{"verified_topics": [{{"group": "...", "topic_id": <int>}}, ...]}}
""".strip()

    resp = client.chat.completions.create(
        model=MODEL_SYNTHESIS,
        messages=[
            {"role": "system", "content": VERIFY_SYSTEM},
            {"role": "user",   "content": prompt},
        ],
        response_format={"type": "json_object"},
    )
    result = json.loads(resp.choices[0].message.content)
    
    original_n = len(source_topics)
    
    insight["source_topics"] = normalize_source_topics(
        result.get("verified_topics", source_topics)
    )
    
    verified_n = len(insight.get("source_topics", []))
    if verified_n != original_n:
        print(f"Adjusted source_topics: {title[:80]} | {original_n} -> {verified_n}")
    if verified_n == 0 and original_n > 0:
        print(f"WARNING: all source_topics removed: {title[:80]}")
    
    return insight

for key in ["key_insights", "appendix"]:
    insights_data[key] = [
        verify_source_topics(i, labels_df, GROUPBY_FIELD)
        for i in insights_data.get(key, [])
    ]

for cat in insights_data.get("by_group", {}):
    insights_data["by_group"][cat] = [
        verify_source_topics(i, labels_df, GROUPBY_FIELD)
        for i in insights_data["by_group"][cat]
    ]

# NOTE: "by_group" is a temporary legacy key name retained in this refactor
# to avoid unnecessary downstream schema changes. Do not rename it in this pass.
for key in ["key_insights", "appendix"]:
    before = len(insights_data.get(key, []))
    insights_data[key] = [i for i in insights_data.get(key, []) if i.get("source_topics")]
    dropped = before - len(insights_data[key])
    if dropped:
        print(f"WARNING: {key}: dropped {dropped} insights with no verified source topics")

for cat in list(insights_data.get("by_group", {}).keys()):
    before = len(insights_data["by_group"][cat])
    insights_data["by_group"][cat] = [
        i for i in insights_data["by_group"][cat] if i.get("source_topics")
    ]
    dropped = before - len(insights_data["by_group"][cat])
    if dropped:
        print(f"WARNING: by_category[{cat}]: dropped {dropped} insights")

with open(OUT("analysis", "insights_structured.json"), "w") as f:
    json.dump(insights_data, f, indent=2)

print("Verification complete.")


Adjusted source_topics: Teachers are building mental-health infrastructure through ordinary supply categ | 6 -> 5
Adjusted source_topics: Flexible seating is really two separate markets: agency furniture and therapeuti | 5 -> 3
Adjusted source_topics: Book requests are increasingly about intervention and belonging, not just buildi | 5 -> 4
Adjusted source_topics: Assistive technology demand is being hidden inside general device requests. | 4 -> 2
Adjusted source_topics: Teachers are turning basic classroom operations into donor asks because continui | 4 -> 2
Adjusted source_topics: Professional development demand is concentrating around named, transferable inst | 5 -> 3
Adjusted source_topics: Food and hygiene requests are really classroom-readiness requests. | 4 -> 3
Adjusted source_topics: Teachers are using physical space itself as an instructional and social tool. | 5 -> 4
Adjusted source_topics: Play-based materials are being defended as serious instruction because teachers  | 4 -

In [31]:
# ── BUILD DOCX + TRACEABILITY ──────────────────────────────────────────────
assert "project_topic_bridge_df" in dir() and not project_topic_bridge_df.empty, (
    "project_topic_bridge_df missing — ensure Cell 13 ran successfully"
)

def get_project_ids(source_topics, bridge_df, max_ids=None):
    if not source_topics:
        return []
    records = []
    for src in source_topics:
        if isinstance(src, dict):
            group = src.get("group", "")
            tid   = str(src.get("topic_id", ""))
        elif isinstance(src, str) and "|" in src:
            group, tid = src.rsplit("|", 1)
        else:
            continue
        topic_key = f"{GROUPBY_FIELD}={group}|topic={tid}"
        matches   = bridge_df[bridge_df["topic_key"] == topic_key]["project_id"].tolist()
        records.extend(matches)
    seen = list(dict.fromkeys(records))
    return seen if max_ids is None else seen[:max_ids]

def source_topics_to_text(source_topics):
    parts = []
    for src in source_topics or []:
        if isinstance(src, dict):
            parts.append(f"{src.get('group','')}|{src.get('topic_id','')}")
        else:
            parts.append(str(src))
    return "; ".join(parts)

all_insights = (
    [("Key Insights", i) for i in insights_data.get("key_insights", [])]
    + [("Appendix",   i) for i in insights_data.get("appendix", [])]
    + [(cat, i)
       for cat, items in insights_data.get("by_group", {}).items()
       for i in items]
)

CSV_MAX_IDS_PER_INSIGHT = CFG["output"]["csv_max_ids_per_insight"]

traceability_rows = []
for section, insight in all_insights:
    pids = get_project_ids(
        insight.get("source_topics", []),
        project_topic_bridge_df,
        max_ids=CSV_MAX_IDS_PER_INSIGHT,
    )
    for pid in pids:
        traceability_rows.append({
            "section":       section,
            "insight_title": insight.get("title", ""),
            "source_topics": source_topics_to_text(insight.get("source_topics", [])),
            "project_id":    pid,
        })

traceability_df = pd.DataFrame(traceability_rows)
traceability_path = OUT("analysis", "insights_traceability.csv")
traceability_df.to_csv(traceability_path, index=False)
print(f"Traceability rows: {len(traceability_df):,}  →  {traceability_path}")

summary_rows = []
for section, insight in all_insights:
    title         = insight.get("title", "")
    source_topics = insight.get("source_topics", [])

    pids       = get_project_ids(source_topics, project_topic_bridge_df, max_ids=None)
    n_projects = len(pids)

    topic_details = []
    for src in source_topics:
        if isinstance(src, dict):
            group = src.get("group", "")
            tid   = str(src.get("topic_id", ""))
        elif isinstance(src, str) and "|" in src:
            group, tid = src.rsplit("|", 1)
        else:
            continue

        match = labels_df[
            (labels_df[GROUPBY_FIELD] == group) &
            (labels_df["topic_id"]    == int(tid))
        ]
        if not match.empty:
            row = match.iloc[0]
            topic_details.append({
                "group":     group,
                "topic_id":  tid,
                "label":     row["proposed_label"],
                "coherence": row["coherence_flag"],
            })
        else:
            topic_details.append({
                "group":     group,
                "topic_id":  tid,
                "label":     "[not found in labels_df]",
                "coherence": "unknown",
            })

    summary_rows.append({
        "section":       section,
        "insight_title": title,
        "n_projects":    n_projects,
        "n_topics":      len(source_topics),
        "topic_details": topic_details,
    })

for row in summary_rows:
    flag = "  ⚠" if any(t["coherence"] != "coherent" for t in row["topic_details"]) else ""
    print(f"{'─'*80}")
    print(f"[{row['section']}]  {row['insight_title']}{flag}")
    print(f"  Projects: {row['n_projects']:,}   |   Source topics: {row['n_topics']}")
    for t in row["topic_details"]:
        coherence_note = f"  ⚠ {t['coherence']}" if t["coherence"] != "coherent" else ""
        print(f"    • {t['group']} / topic {t['topic_id']}: {t['label']}{coherence_note}")
    print()

flat_rows = []
for row in summary_rows:
    insight_total = row["n_projects"]

    for t in row["topic_details"]:
        topic_pids = get_project_ids(
            [{"group": t["group"], "topic_id": t["topic_id"]}],
            project_topic_bridge_df,
            max_ids=None,
        )
        flat_rows.append({
            "section":        row["section"],
            "insight_title":  row["insight_title"],
            "n_projects_insight": insight_total,
            "n_projects_topic":   len(topic_pids),
            "n_topics":       row["n_topics"],
            "group":          t["group"],
            "topic_id":       t["topic_id"],
            "topic_label":    t["label"],
            "coherence":      t["coherence"],
        })

summary_df = pd.DataFrame(flat_rows)
summary_path = OUT("analysis", "insights_topic_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"Summary rows: {len(summary_df):,}  →  {summary_path}")

def add_heading(doc, text, level):
    p = doc.add_heading(text, level=level)
    p.runs[0].font.color.rgb = RGBColor(0, 0, 0)
    return p

def add_insight(doc, insight):
    p = doc.add_paragraph()
    run = p.add_run(insight.get("title", ""))
    run.bold = True
    run.font.size = Pt(11)

    for label, key in [
        ("What we're seeing:",          "what_seeing"),
        ("Why this is easy to miss:",   "why_easy_to_miss"),
        ("What this means for funders:","what_it_means"),
    ]:
        p = doc.add_paragraph()
        label_run = p.add_run(label + "  ")
        label_run.bold = True
        label_run.font.size = Pt(10)
        body_run = p.add_run(insight.get(key, ""))
        body_run.font.size = Pt(10)
        p.paragraph_format.space_after = Pt(2)

    doc.add_paragraph()

doc = Document()

for section in doc.sections:
    section.top_margin    = Inches(1)
    section.bottom_margin = Inches(1)
    section.left_margin   = Inches(1)
    section.right_margin  = Inches(1)

style = doc.styles["Normal"]
style.font.name = "Arial"
style.font.size = Pt(10)

add_heading(doc, "Investigated Across Project Categories", level=1)
add_heading(doc, "Key Insights", level=2)

for insight in insights_data.get("key_insights", []):
    add_insight(doc, insight)

if insights_data.get("appendix"):
    add_heading(doc, "Appendix", level=2)
    for insight in insights_data["appendix"]:
        add_insight(doc, insight)

add_heading(doc, "By Category", level=1)

for category, items in insights_data.get("by_group", {}).items():
    add_heading(doc, category, level=2)
    for insight in items:
        add_insight(doc, insight)

docx_path = OUT("analysis", "R_I_Trends_generated.docx")
doc.save(docx_path)
print(f"Docx saved → {docx_path}")
print(f"Open and compare against R_I_Trends__Prototype_Checkpoint__1_.docx")


Traceability rows: 13,851  →  /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/project_category/2026-03-27/analysis/insights_traceability.csv
────────────────────────────────────────────────────────────────────────────────
[Key Insights]  Teachers are building mental-health infrastructure through ordinary supply categories.
  Projects: 32,230   |   Source topics: 5
    • Flexible Seating / topic 5: Calm Down Corners for Social-Emotional Regulation in Special Education
    • Educational Kits & Games / topic 2: Calm Corner Sensory Tools for Emotional Regulation
    • Books / topic 9: Social Emotional Learning (SEL) Tools for Emotional Regulation in Early Childhood
    • Sports & Exercise Equipment / topic 7: Sensory Tools for Emotional Regulation in Prekindergarten and Kindergarten Classrooms
    • Art Supplies / topic 6: Social Emotional Learning Resources for Stress Regulation

────────────────────────────────────────────────────────────────────────────────
[Key Insi